# Pyspark SCD 2 type :-
##      1) Using Hash function to check

In [0]:
# Generating Sample Data Sample initial data
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("PysparkSCD2").getOrCreate()
initial_df = spark.createDataFrame(
    [(1, "Widget A", 10.00),
     (2, "Widget B", 12.50),
     (3, "Widget C", 20.00)],
    ["id", "name", "price"]
)
initial_df.show()
initial_df.printSchema()

**Adding a column for merge condition,That will be key column to insert or update
**

In [0]:
from pyspark.sql.functions import sha2,lit,concat_ws,col
initial_df=initial_df.withColumn("new_hash",sha2(concat_ws(lit('_'),col("id"),col("name")),256))
initial_df.show(truncate=False)

In [0]:
# Importing current_timestamp package 
from pyspark.sql.functions import current_timestamp

In [0]:
#Dropping table if exists
spark.sql("Drop table main.demo_schema.scd2_table_demo")

In [0]:
# 
spark.sql("""
          Create table main.demo_schema.scd2_table_demo 
           (
          new_hash string  NOT NULL PRIMARY KEY,
          id int,
          name string,
          price double,
          effective_date timestamp,
          end_date timestamp
          ) """)

Others ways to create a delta table from dataframe
**df.write.format("delta").saveAsTable("main.demo_schema.scd2_table_demo")**

In [0]:
# Describe the table
spark.sql("DESCRIBE  main.demo_schema.scd2_table_demo").display()

In [0]:
# Extended describe
spark.sql("DESCRIBE EXTENDED main.demo_schema.scd2_table_demo").display()

In [0]:
spark.read.table("main.demo_schema.scd2_table_demo").display()

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import lit,current_timestamp
from pyspark.sql.types import StringType,TimestampType,IntegerType,DoubleType
dt= DeltaTable.forName(spark,"main.demo_schema.scd2_table_demo")


In [0]:
# update Current row
dt.alias("t").\
    merge(
        source=initial_df.alias("s"),
        condition="s.new_hash=t.new_hash and t.end_date is null"
    ).\
        whenMatchedUpdate(
            set ={"t.end_date":current_timestamp()}
                          ).\
                        execute()

In [0]:
spark.read.table("main.demo_schema.scd2_table_demo").display()

In [0]:
initial_df=initial_df.\
    withColumn("effective_date",current_timestamp()).\
    withColumn("end_date",lit(None).cast(TimestampType())).\
    select(initial_df.new_hash,initial_df.id.cast(IntegerType()),initial_df.name.cast(StringType()),initial_df.price.cast(DoubleType()),"effective_date","end_date").\
    write.\
    mode("append").\
    saveAsTable("main.demo_schema.scd2_table_demo")


In [0]:
spark.read.table("main.demo_schema.scd2_table_demo").display()

In [0]:
# Sample initial data
initial_upd = spark.createDataFrame(
    [
     (2, "Widget B",15)],
    ["id", "name", "price"]
)
initial_upd=initial_upd.withColumn("new_hash",sha2(concat_ws(lit('_'),col("id"),col("name")),256))
initial_upd.show()

In [0]:
# update Current row
dt.alias("t").\
    merge(
        source=initial_upd.alias("s"),
        condition="s.new_hash=t.new_hash and t.end_date is null"
    ).\
        whenMatchedUpdate(
            set ={"t.end_date":current_timestamp()}
                          ).\
                        execute()

In [0]:
spark.read.table("main.demo_schema.scd2_table_demo").display()

In [0]:
initial_upd=initial_upd.withColumn("effective_date",current_timestamp()).withColumn("end_date",lit(None).cast(TimestampType())).select(initial_upd.new_hash,initial_upd.id.cast(IntegerType()),initial_upd.name.cast(StringType()),initial_upd.price.cast(DoubleType()),"effective_date","end_date").write.mode("append").saveAsTable("main.demo_schema.scd2_table_demo")


In [0]:
spark.read.table("main.demo_schema.scd2_table_demo").display()

In [0]:
# Sample initial data
initial_upd1 = spark.createDataFrame(
    [
     (1, "Widget BA",15)],
    ["id", "name", "price"]
)
initial_upd1=initial_upd1.withColumn("new_hash",sha2(concat_ws(lit('_'),col("id"),col("name")),256))
initial_upd1.show()

In [0]:
# update Current row
dt.alias("t").\
    merge(
        source=initial_upd1.alias("s"),
        condition="s.new_hash=t.new_hash and t.end_date is null"
    ).\
        whenMatchedUpdate(
            set ={"t.end_date":current_timestamp()}
                          ).\
        whenNotMatchedInsert(
            values={
             "t.new_hash": "s.new_hash",
             "t.id": "s.id",
             "t.name": "s.name",
             "t.price": "s.price",
             "t.effective_date": current_timestamp(),
             "t.end_date": lit(None).cast(TimestampType())
        }
            ).\
                        execute()

In [0]:
spark.read.table("main.demo_schema.scd2_table_demo").display()

In [0]:
spark.read.table("main.demo_schema.scd2_table_demo").explain()

In [0]:
spark.sql("alter table main.demo_schema.scd2_table_demo set TBLPROPERTIES( delta.enableChangeDataFeed=True)")

In [0]:
spark.read.table("main.demo_schema.scd2_table_demo").display()

In [0]:
# Sample initial data
initial_del1 = spark.createDataFrame(
    [
     (1, "Widget BA",15)],
    ["id", "name", "price"]
)
initial_del1=initial_del1.withColumn("new_hash",sha2(concat_ws(lit('_'),col("id"),col("name")),256))
initial_del1.show()

In [0]:
# delete   row
dt.alias("t").\
    merge(
        source=initial_del1.alias("s"),
        condition="s.new_hash=t.new_hash"
    ).\
    whenMatchedDelete().\
                        execute()

In [0]:
spark.read.table("main.demo_schema.scd2_table_demo").display()